In [12]:
import pandas as pd
import numpy as np

In [61]:
curve_df = pd.read_hdf('./data/data.h5', key='curve_data')
sample_info = pd.read_hdf('./data/data.h5', key='sample_info')
igi_gene_call = pd.read_hdf('./data/data.h5', key='igi_gene_call')

In [62]:
max_cycle_df = curve_df.groupby(['curve_idx']).cycle_no.agg(max).reset_index()
luner_ids = max_cycle_df[max_cycle_df.cycle_no == 45].curve_idx.unique()
thermo_ids = max_cycle_df[max_cycle_df.cycle_no == 40].curve_idx.unique()

In [63]:
tail_df = (curve_df
            .loc[((curve_df.curve_idx.isin(luner_ids)) & (curve_df.cycle_no >= 40)) | 
                 ((curve_df.curve_idx.isin(thermo_ids)) & (curve_df.cycle_no >= 35))])

In [73]:
drn_threshold_df = curve_df[['curve_idx','threshold']].drop_duplicates()
drn_threshold_df.loc[:,'drn_low'] = drn_threshold_df.threshold - 0.05*drn_threshold_df.threshold
drn_threshold_df.loc[:,'drn_high'] = drn_threshold_df.threshold + 0.05*drn_threshold_df.threshold

drn_threshold_df = drn_threshold_df.drop('threshold', axis=1)

In [105]:
drn_amb_df = (tail_df
                .merge(drn_threshold_df, how='inner', on='curve_idx'))
amb_ids = (drn_amb_df
               .loc[(drn_amb_df.drn <= drn_amb_df.drn_high) &
                    (drn_amb_df.drn >= drn_amb_df.drn_low)]
               .curve_idx.unique())

print('Number of identified curves: ', len(amb_ids))

Number of identified curves:  821


In [106]:
join_df = (curve_df
            .loc[curve_df.curve_idx.isin(amb_ids)]
            .merge(sample_info, how='inner', on=['well_position','pcr_plate'])
            .merge(igi_gene_call, how='inner', on=['pcr_plate','sample_id','target']))

In [114]:
(join_df
    .replace({np.nan: 'Imputed-Invalid'})
    .groupby(['sample_type','igi_call','current_sample_result'])
    .curve_idx.nunique())

sample_type                                 igi_call  current_sample_result
Buffer Negative Control (Extraction)        Negative  Imputed-Invalid            7
                                                      Negative                   3
                                            Positive  Negative                   1
Clinical Sample                             Negative  Inconclusive              29
                                                      Invalid                   85
                                                      Negative                 309
                                                      Positive                  46
                                            Positive  Inconclusive              36
                                                      Invalid                   20
                                                      Negative                  56
                                                      Positive                  17
Human Norma

In [117]:
curve_df[curve_df.curve_idx.isin(amb_ids)].to_csv('./data/amb_curves.csv', index=False)